In [2]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict

META_COLS = ("dataset", "task_id", "best_model")

def load(path):
    df = pd.read_csv(path)
    feats = [c for c in df if c not in META_COLS and not c.startswith("score_")]
    return df, feats

def make_meta_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                               class_weight="balanced_subsample", random_state=0),
    )

clf_df, clf_feats = load("../data/meta_clf.csv")
reg_df, reg_feats = load("../data/meta_reg.csv")

In [3]:
import numpy as np, pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
# reuses load(), make_meta_model(), clf_df/clf_feats, reg_df/reg_feats from Notebook 2

def regret_eval(df, feats, name):
    X, y = df[feats], df["best_model"]
    classes = np.sort(y.unique())
    proba = cross_val_predict(make_meta_model(), X, y, cv=LeaveOneOut(), method="predict_proba")

    models = [c[6:] for c in df if c.startswith("score_")]
    S = np.clip(df[[f"score_{m}" for m in models]].values, -1, None)  # clip R² outliers
    col = {m: i for i, m in enumerate(models)}
    idx = np.array([col[c] for c in classes])      # meta-model classes -> score columns
    best, r = S.max(1), np.arange(len(S))

    top1 = idx[proba.argmax(1)]
    top3 = idx[np.argsort(-proba, 1)[:, :3]]
    fixed = S.mean(0).argmax()                      # best single model on average

    picks = {
        "meta top-1":                  S[r, top1],
        "meta top-3 (try 3, keep best)": np.take_along_axis(S, top3, 1).max(1),
        f"always {models[fixed]}":     S[:, fixed],
        "always RandomForest":         S[:, col["RandomForest"]],
        "random pick":                 S.mean(1),
    }
    out = pd.DataFrame({
        k: {"mean regret": (best - v).mean(),
            "median regret": np.median(best - v),
            "within 0.01 of best": ((best - v) <= 0.01).mean()}
        for k, v in picks.items()}).T.round(3)
    rank1 = ((S > S[r, top1][:, None]).sum(1) + 1).mean()
    print(f"\n=== {name} === (mean rank of meta top-1 pick among {len(models)} models: {rank1:.2f}; 1 = best)")
    print(out)

regret_eval(clf_df, clf_feats, "Classification")
regret_eval(reg_df, reg_feats, "Regression")


=== Classification === (mean rank of meta top-1 pick among 7 models: 2.68; 1 = best)
                               mean regret  median regret  within 0.01 of best
meta top-1                           0.034          0.007                0.569
meta top-3 (try 3, keep best)        0.005          0.000                0.903
always HistGB                        0.024          0.007                0.569
always RandomForest                  0.036          0.016                0.389
random pick                          0.067          0.056                0.014

=== Regression === (mean rank of meta top-1 pick among 7 models: 2.32; 1 = best)
                               mean regret  median regret  within 0.01 of best
meta top-1                           0.104          0.004                0.581
meta top-3 (try 3, keep best)        0.030          0.000                0.806
always RandomForest                  0.051          0.023                0.258
random pick                          0.222